In [ ]:
!pip install pysam

In [ ]:
from tqdm import tqdm
import pysam

In [ ]:
chrom = 1
b = 0

In [ ]:
inds_twins = {}

with open(f'/mnt/project/DNM/twins/blocks/twins_pairs_b{b}.txt') as F:
            
    for i, row in enumerate(F):
                
        row = row.strip()
        row = row.split(' ')
    
        twin1 = str(row[0])
        twin2 = str(row[1])
            
        twin1_idx = f'{twin1}_b{b}_p{i}'
        twin2_idx = f'{twin2}_b{b}_p{i}'
            
        inds_twins[twin1_idx] = twin2_idx
        inds_twins[twin2_idx] = twin1_idx

In [ ]:
inds_diffs = {}

path = f'/mnt/project/DNM/twins/diffs (full)/chr{chrom}/'

with open(f'/mnt/project/DNM/twins/blocks/twins_pairs_b{b}.txt') as F:

    for i, _ in enumerate(F):

        # Remove pair if twin is missing
        cnt = 0

        with open(f'{path}twins_chr{chrom}_b{b}_p{i}.tsv') as f:

            for row in f:

                row = row.strip()
                row = row.split()
                cnt += 1 if len(row) > 0 else 0

        if cnt < 2:
            continue

        # Record differences per twin
        with open(f'{path}twins_chr{chrom}_b{b}_p{i}.tsv') as f:

            for row in f:

                row = row.strip()
                row = row.split()

                ind = str(row[0].split('_')[1])
                ind_idx = f'{ind}_b{b}_p{i}'
                if ind_idx not in inds_diffs:
                    inds_diffs[ind_idx] = []

                ind_diffs = row[1].split('|')[1:] if len(row) == 2 else []

                for ind_diff in ind_diffs:
                    inds_diffs[ind_idx].append(ind_diff)

In [ ]:
inds_path = {}

# paths = '/mnt/project/Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]/'

for ind_idx in inds_twins:
    
    ind = str(ind_idx.split('_')[0])
    # inds_path[ind_idx] = f'{paths}{ind[:2]}/{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'
    inds_path[ind_idx] = f'{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'

In [ ]:
inds_diffs_qc = {}
inds_diffs_errors = {}

for _, ind_idx in tqdm(enumerate(inds_diffs)):
    
    # Twin --------------------------------
    
    twin_idx = inds_twins[ind_idx]
    
    try:
        vcf = pysam.VariantFile(inds_path[twin_idx])
    except:
        continue # failed to read gVCF

    twin_info = {}
    
    for diff_id in inds_diffs[ind_idx]:
        
        pos = int(diff_id.split(':')[1])
        ref = diff_id.split(':')[2]
        alt = diff_id.split(':')[3]
            
        for F in vcf.fetch(f'chr{chrom}', pos-1, pos, reopen = True):
            
            for f in list(F.samples):
            
                dp = F.samples[f]['DP']
                gq = F.samples[f]['GQ']
                gt = F.samples[f]['GT']
                
                if None in gt:
                    continue
                
                if F.pos == pos:
                    
                    ref_ad = 0
                    alt_ad = 0
                    
                    if ref in F.alleles:
                        i = F.alleles.index(ref)
                        ref_ad = F.samples[f]['AD'][i]
                        
                    if alt in F.alleles:
                        i = F.alleles.index(alt)
                        alt_ad = F.samples[f]['AD'][i]
                        
                    alleles = (F.alleles[gt[0]], F.alleles[gt[1]])
                        
                else:
                    ref_ad = dp
                    alt_ad = 0
                    alleles = (ref, ref)
                
                twin_info[diff_id] = (ref_ad, alt_ad, dp, gq, alleles)
        
    # Individual --------------------------------
    
    try:
        vcf = pysam.VariantFile(inds_path[ind_idx])
    except:
        continue # failed to read gVCF
    
    inds_diffs_qc[ind_idx] = []
    inds_diffs_errors[ind_idx] = []
    
    for diff_id in inds_diffs[ind_idx]:

        if diff_id not in twin_info:
            continue
            
        pos = int(diff_id.split(':')[1])
        ref = diff_id.split(':')[2]
        alt = diff_id.split(':')[3]
            
        for F in vcf.fetch(f'chr{chrom}', pos-1, pos, reopen = True):
            
            for f in list(F.samples):
            
                dp = F.samples[f]['DP']
                gq = F.samples[f]['GQ']
                gt = F.samples[f]['GT']
                
                if None in gt:
                    continue
                
                if F.pos == pos:
                    
                    ref_ad = 0
                    alt_ad = 0
                    
                    if ref in F.alleles:
                        i = F.alleles.index(ref)
                        ref_ad = F.samples[f]['AD'][i]
                        
                    if alt in F.alleles:
                        i = F.alleles.index(alt)
                        alt_ad = F.samples[f]['AD'][i]
                        
                    alleles = (F.alleles[gt[0]], F.alleles[gt[1]])
                        
                else:
                    ref_ad = dp
                    alt_ad = 0
                    alleles = (ref, ref)
                    
                info = (diff_id, ref_ad, alt_ad, dp, gq, alleles,
                        twin_info[diff_id][0], twin_info[diff_id][1], twin_info[diff_id][2], twin_info[diff_id][3], twin_info[diff_id][4])
                        
                if ((ref in alleles and alt in alleles) and
                    (twin_info[diff_id][4][0] == twin_info[diff_id][4][1] and ref in twin_info[diff_id][4])):
                    inds_diffs_qc[ind_idx].append(info)
                else:
                    inds_diffs_errors[ind_idx].append(info)

In [ ]:
with open(f'twins_qc_chr{chrom}_b{b}.txt', 'w') as f:
    for ind_idx in inds_diffs_qc:
        for (diff_id, ref_ad, alt_ad, dp, gq, alleles, twin_ref_ad, twin_alt_ad, twin_dp, twin_gq, twin_alleles) in inds_diffs_qc[ind_idx]:
            f.write(f'{ind_idx} {diff_id} {ref_ad} {alt_ad} {dp} {gq} {alleles} {twin_ref_ad} {twin_alt_ad} {twin_dp} {twin_gq} {twin_alleles}\n')

In [ ]:
with open(f'twins_errors_chr{chrom}_b{b}.txt', 'w') as f:
    for ind_idx in inds_diffs_errors:
        for (diff_id, ref_ad, alt_ad, dp, gq, alleles, twin_ref_ad, twin_alt_ad, twin_dp, twin_gq, twin_alleles) in inds_diffs_errors[ind_idx]:
            f.write(f'{ind_idx} {diff_id} {ref_ad} {alt_ad} {dp} {gq} {alleles} {twin_ref_ad} {twin_alt_ad} {twin_dp} {twin_gq} {twin_alleles}\n')